# Chapter 5 Predicting Revenue with Linear Regression

In [ ]:
#%pip install statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import plotly.io as pio
#pio.renderers.default = "notebook"
import plotly.express as px
import statsmodels as sm
import datetime as dt

In [ ]:
ad = pd.read_csv('https://raw.githubusercontent.com/librowndev/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter05/Datasets/advertising.csv')

In [ ]:
ad.head()

In [ ]:
fig = px.scatter(data_frame=ad,
                x='TV',
                y='Sales')

fig.show(renderer="notebook")


In [ ]:
import plotly.express as px

fig = px.scatter(
    ad, x='TV', y='Sales',
    trendline='ols',
    trendline_color_override='red'
    
)
fig.show(renderer='notebook')


In [ ]:
plt.scatter(data=ad, x='TV', y='Sales')
plt.show()

## Exercise 5.02 Creating Features for Customer Revenue Prediction

Approach: Build a model to predict the revenue of the customers in 2020 based on purchases made in 2019. The 2020 purchase data can then be used to predict customer revenue for 2021.

Goals for the exercises are to manipulate the data and create variables that will model the customer spend for 2020, based on past activity. Total customer spend for 2020 will be the dependent variable. Independent variables will be the features that captue information about the customer's past behavior. This will also require aggregation of the data in order to get one record for each customer.

These are items for feature engineering:

    - revenue_2019 (total revenue for 2019)
    - days_since_first_purchase (number of days since first customer purchase)
    - days_since_last_purchase (number of days since most recent purchase)
    - number_of_purchases (total number of purchases by customer in 2019)
    - avg_order_cost (average value of orders placed by customer in 2019)
    - revenue_2020 (total revenue for 2020)


In [ ]:
azra = pd.read_csv("https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter05/Datasets/azra_retail_transactions.csv")

In [ ]:
azra.head()

In [ ]:
print(azra.shape)
print(azra.info())
azra.describe()

In [ ]:
# Convert InvoiceDate to datetime
azra['InvoiceDate'] = pd.to_datetime(azra['InvoiceDate'])

# add a revenue column
azra['revenue'] = azra['UnitPrice'] * azra['Quantity']

#check results
azra.head()

Each invoice can be spread over multiple rows, one for each time of product (line item) and a customer can buy multiple products in an order. The data needs to be aggregated by InvoiceNo for each transaction to be on a single row. First, have to specify how to combine the rows using the agg() function.

The InvoiceDate and CustomerID will be the same for all rows for the same invoice, therefore, the first entry is chosed. Revenue is summed across all items for the same invoice to obtain the total for that invoice.

In [ ]:
operations = {'revenue': 'sum',
               'InvoiceDate': 'first',
                'CustomerID': 'first'}

azra = azra.groupby('InvoiceNo').agg(operations)
azra.head()

In [ ]:
# Create year columns in the data
azra['year'] = azra['InvoiceDate'].dt.year

#calculate the difference between the end of 2019 and the invoice date
azra['days_since']= (dt.datetime(year=2019, month=12, day=31) - azra['InvoiceDate']).apply(lambda x: x.days)
azra.head()

Next will be creating the features for days since first and last purchase, and the number of purchased and total revenue for 2019. Again, the operations will be defined with aggregation functions and applied by grouping the CustomerID.

For days_since, the calculations are for the max and min number of days, as well as the number of unique values (the number of days the customer made a purchase). 

These features will be predictors and will be stored in the variable **X**.


In [ ]:
#feature_ops = {'revenue_2019': 'sum',
#             'days_since': ['max', 'min', 'nunique']}

#X = azra[azra['year']==2019].groupby('CustomerID').agg(feature_ops)
X = (azra[azra['year'] == 2019]
     .groupby('CustomerID')
     .agg(revenue_2019=('revenue', 'sum'),
          days_since_first=('days_since', 'max'),
          days_since_last=('days_since', 'min'),
          days_since_nunique=('days_since', 'nunique'))
     .reset_index())

X.head()

In [ ]:
# reset the names of the columns to eliminate the multi-level labels
#X.columns = [' '.join(col).strip() for col in X.columns.values]

X.head()

In [ ]:
# calculate the average spend per order
X['avg_order_cost']= X['revenue_2019'] / X['days_since_nunique']
X.head()

In [ ]:
# predictors are the revenue for 2020, store them in variable y

y = azra[azra['year']==2020].groupby('CustomerID')['revenue'].sum().reset_index()
y.columns =['CustomerID', 'revenue_2020']
#y.drop('CustomerID', axis=1, inplace=True)
y.head()

In [ ]:
w2 = pd.merge(X, y, how='outer', on='CustomerID')
w2.isna().sum()

In [ ]:
# concat predictors and outcomes into one dataframe, change to merge

wrangled_df = pd.merge(X, y, how='outer', on='CustomerID')
#wrangled_df.columns =['revenue_2019', 'days_since_first_purchase', 'days_since_last_purchase', 'number_of_purchases','avg_order_cost', 'revenue_2020']
#wrangled_df.set_index(CustomerID)

wrangled_df.head()

In [ ]:
wrangled_df.isna().sum()

In [ ]:
# drop rows where either revenue is null

wrangled_df = wrangled_df[~wrangled_df.revenue_2019.isnull()]
wrangled_df = wrangled_df[~wrangled_df.revenue_2020.isnull()]
wrangled_df.isna().sum()

In [ ]:
# as best practice save the file (will not save to iPad)

# wrangled_df.to_csv('wrangled_data.csv')
#rename days_since_ununique to number_of_purchases
wrangled_df = wrangled_df.rename(columns={'days_since_nunique': 'number_of_purchases'})

wrangled_df.head()

### Assessing Features Using Visualizations and Correlations

After creating features on interest and data wrangling, next step is assess the features. Perform sanity checks, understanding indiividual columns better, understand associations between features and patterns in the data.

1. Perform sanity checks: make sure values are what is expected. Plot histograms
2. Create scatter plots
3. Assess correlations

### Exercise 5.03 Examining Relationships Between Predictors and Outcomes

In [ ]:
# use wrangled_df from previous exercise (or read in csv)
# plotting with matplotlib, but may need plotly
#wrangled_df.plot.scatter(x='days_since_first_purchase', y='revenue_2020',color='blue')
#plt.show()
fig = px.scatter(data_frame=wrangled_df, x='days_since_first', y='revenue_2020')

fig.show(renderer='notebook')

In [ ]:
# lot at pairplots (scatter matrix in plotly)

sns.pairplot(wrangled_df)
plt.show()

In [ ]:
#limit data to pairplot row for the target variable
sns.pairplot(wrangled_df, x_vars=wrangled_df.columns, y_vars='revenue_2020')
plt.show()

In [ ]:
# Quantify associations with corr() functions
wrangled_df.corr()

## Performing and Interpreting Linear Regression


### Exercise 5.04 Building a Linear model Predicting Customer Spend

In [ ]:
wrangled_df.head()

In [ ]:
X1 = wrangled_df[['revenue_2019', 'days_since_last','number_of_purchases','avg_order_cost']]
#X1.set_index('CustomerID')
X1.head()

In [ ]:
y1 = wrangled_df['revenue_2020']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X1, y1, random_state=100)

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train, y_train)

print(f"model coefficients are:  {lr.coef_} ")
print(f"model intercept is  {lr.intercept_}")



In [ ]:
# predict correlation between predictual and actual revenue_2020 values
np.corrcoef(lr.predict(X_test), y_test)

### Notes

1. Coefficients and intercept are different from book values, but correlation is similar

2. chapter has 2 Activities that use a different dataset and work through the entire process, will be done in a separate notebook.